# Samaritan solver on an A100 — Qwen3.8-27B (turnkey)

Serves **Qwen3.8-27B** (Q8_0 GGUF) on this Colab A100 via **Ollama**, and
exposes it as an OpenAI-compatible endpoint your **local** Samaritan harness
points at — no code change, just `SAMARITAN_URL`.

**Why Ollama, not vLLM:** vLLM can't run FP8 MoE on the A100 (Ampere has no
FP8 tensor cores), and this model's newer arch isn't confirmed on the vLLM
build here. The GGUF + Ollama path is proven (11.5M downloads) and bundles
its own CUDA, so there's no build fight with Colab's CUDA-13. We use the
**MTP build** (`-mtp-`), which carries a built-in multi-token-prediction draft
head for *self-speculative decoding* — faster generation at the same quality.
Q8_0 (~30 GB) fits the 40 GB card with full GPU offload.

**Before running:** Runtime → Change runtime type → **A100 GPU** (Pro+).
Then Runtime → **Run all**. The tunnel cell prints the line to paste locally.

**Honest limits:** Colab is not 24/7 (Pro+ background ~24 h, can drop) — good
for eval pushes and self-training, not a deployment. **Ollama has no API-key**
auth, so the public tunnel URL is the *only* guard — don't share it, and run
the shutdown cell (or stop the runtime) when done; a forgotten A100 burns
compute units.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. Install Ollama and start the server

Ollama ships its own CUDA runtime, so nothing compiles against Colab's stack.
Its installer extracts a **zstd**-compressed tarball, and Colab's image doesn't
ship `zstd` — so we `apt-get` it first (otherwise the install aborts and there's
no `ollama` on PATH). We also wipe any partial prior install and **assert the
`llama-server` runner landed**, not just the daemon binary — a partial extract
leaves the daemon serving `/v1` but 500-ing at generation. Colab has no systemd,
so we start the daemon ourselves and keep the model resident
(`OLLAMA_KEEP_ALIVE=-1`) so eval runs don't reload it.

In [ ]:
import os, subprocess, time, urllib.request, glob, shutil
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
!rm -rf /usr/local/lib/ollama   # drop any partial prior extraction so the runner lands too
!curl -fsSL https://ollama.com/install.sh | sh
# Ollama needs BOTH the daemon binary AND its llama-server runner under
# /usr/local/lib/ollama. A partial extract leaves the daemon serving /v1 but
# 500-ing at generation ('llama-server binary not found') — so assert both.
ob = shutil.which('ollama')
runner = glob.glob('/usr/local/lib/ollama/**/llama-server', recursive=True)
assert ob and runner, f'incomplete Ollama install — ollama={ob}, runner={runner}; see output above'
print('ollama + runner OK:', runner[0])
env = {**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434', 'OLLAMA_KEEP_ALIVE': '-1'}
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'),
                       stderr=subprocess.STDOUT, env=env)
up = False
for _ in range(30):
    try:
        if urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=3).status == 200:
            up = True; break
    except Exception:
        time.sleep(2)
print('ollama daemon:', 'UP' if up else 'not up yet — see ollama.log')

## 3. Pull Qwen3.8-27B (Q8_0, MTP) and alias it to `samaritan-playout`

`qwen3.8:27b-mtp-q8_0` is the ~30 GB Q8_0 build **with the MTP draft head**
(self-speculative decoding — the model drafts its own next few tokens, which
Ollama verifies in one pass; `draft_num_predict` defaults to 4). Same size and
quality as plain Q8_0, just faster generation. Fits the 40 GB A100 with room.
There is no Q6_K in the Ollama library for the 27B and bf16 (56 GB) won't fit,
so Q8_0 is the tag; the plain `qwen3.8:27b-q8_0` (no MTP) is the fallback if
the MTP build misbehaves. We copy it to `samaritan-playout` (the name the
harness asks for) and bake a 16k context in via a Modelfile — this model
**thinks hard by default**, so we don't want traces truncated. 16k is safe on
40 GB; raise it with headroom, lower it if you OOM. The pull is ~30 GB.

In [ ]:
TAG = 'qwen3.8:27b-mtp-q8_0'   # plain (no MTP) fallback: 'qwen3.8:27b-q8_0'
import subprocess
subprocess.run(['ollama', 'pull', TAG], check=True)
with open('Modelfile', 'w') as f:
    f.write(f'FROM {TAG}\nPARAMETER num_ctx 16384\n')
subprocess.run(['ollama', 'create', 'samaritan-playout', '-f', 'Modelfile'], check=True)
!ollama list

## 4. Expose it with a cloudflared tunnel

Publishes Ollama's port 11434 as a public `https://…trycloudflare.com` URL.
The **`--http-host-header localhost:11434`** flag is required: Ollama validates
the Host header (DNS-rebinding protection) and returns **403** for the public
cloudflare hostname — this rewrites Host to localhost before it reaches Ollama.
**Reminder:** Ollama has no auth, so this URL is the only thing protecting the
endpoint — treat it like a secret and stop the runtime when you're done.

In [ ]:
import subprocess, re, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
cf = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434',
                       '--http-host-header', 'localhost:11434'],
                      stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)
public = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cf.log').read())
    if m: public = m.group(0); break
if not public:
    print('no tunnel URL yet — tail of cf.log:'); print(open('cf.log').read()[-1500:])
else:
    print('Tunnel up. Paste into your LOCAL terminal:\n')
    print(f'  $env:SAMARITAN_URL = "{public}/v1"                 # PowerShell')
    print(f'  $env:SAMARITAN_API_KEY = "ollama"                    # any value; Ollama ignores it')
    print(f'  $env:SAMARITAN_MODEL = "samaritan-playout:latest"    # Ollama tags models; match it exactly')
    print(f'\n  export SAMARITAN_URL="{public}/v1"                   # bash')
    print(f'  export SAMARITAN_API_KEY="ollama"')
    print(f'  export SAMARITAN_MODEL="samaritan-playout:latest"')

## 5. Self-test + token speed

Hits Ollama's **native** `/api/chat` *through the tunnel*, so one call (a) proves
the public URL works end to end, (b) warms the model into VRAM, and (c) reports
real throughput. `eval_duration` / `prompt_eval_duration` are measured on the
*server*, so the tok/s figures are clean decode/prefill speed — unaffected by
tunnel latency (only *wall* includes that). The first call also loads ~30 GB, so
give it a minute; `load_duration` is reported separately and excluded from the
speed numbers. The reply tail may be mid-`<think>` (we cap it at 512 tokens just
to measure) — that's fine, the harness strips thinking.

In [ ]:
import urllib.request, json, time
payload = json.dumps({
    'model': 'samaritan-playout',
    'messages': [{'role': 'user', 'content': 'Think step by step, then give the number: what is 128 * 47?'}],
    'stream': False,
    'options': {'num_predict': 512, 'temperature': 1.0, 'top_p': 0.95, 'top_k': 20},
}).encode()
req = urllib.request.Request(f'{public}/api/chat', data=payload,
    headers={'Content-Type': 'application/json'})
t0 = time.time()
r = json.loads(urllib.request.urlopen(req, timeout=600).read())
wall = time.time() - t0
gen_n, gen_s = r.get('eval_count', 0), r.get('eval_duration', 1) / 1e9
pp_n,  pp_s  = r.get('prompt_eval_count', 0), r.get('prompt_eval_duration', 1) / 1e9
load_s = r.get('load_duration', 0) / 1e9
print(f'generation : {gen_n:>4} tok / {gen_s:5.1f}s = {gen_n/max(gen_s,1e-9):6.1f} tok/s   <- MTP self-speculation')
print(f'prefill    : {pp_n:>4} tok / {pp_s:5.2f}s = {pp_n/max(pp_s,1e-9):6.0f} tok/s')
print(f'model load : {load_s:5.1f}s (first call only)   |   wall (incl. tunnel): {wall:.1f}s')
print('--- reply tail ---')
print(r['message']['content'][-400:])

# To A/B whether MTP actually helps, pull the plain tag and re-measure:
#   subprocess.run(['ollama','pull','qwen3.8:27b-q8_0'], check=True)
#   open('Mp','w').write('FROM qwen3.8:27b-q8_0\nPARAMETER num_ctx 16384\n')
#   subprocess.run(['ollama','create','plain','-f','Mp'], check=True)
# then rerun this cell with model='plain'; MTP should show higher gen tok/s.

## Now, on your laptop

```powershell
$env:SAMARITAN_URL = "https://<random>.trycloudflare.com/v1"   # from cell 4
$env:SAMARITAN_API_KEY = "ollama"                              # any value
$env:SAMARITAN_MODEL = "samaritan-playout:latest"              # Ollama needs the tag
$env:MAX_TOKENS = "8192"      # Qwen3.8 thinks a lot — give the trace room
$env:DATASET = "$env:USERPROFILE\models\reasoning\gsm-symbolic-p2.jsonl"
cargo run -p samaritan-run --example reason_eval
```

(`SAMARITAN_MODEL` matters because Ollama tags every model `name:latest` and
its OpenAI endpoint matches the id exactly — the bare `samaritan-playout` 404s.
The local llama.cpp server uses the bare alias, so leave it unset for that.)

**Sampling note:** Qwen recommends **temp 1.0, top_p 0.95, top_k 20, repeat 1.0**
for this model in thinking mode. `reason_eval` currently sends temp 0.6 / repeat
1.1 (tuned for the local 4B) — fine for a first read, but ask me to make the
example's sampling env-configurable and you can run this model at its own
recommended settings.

Leave this notebook running. This is the first real read from a *capable*
substrate — it should clear the local 4B's 2/10 on GSM-Symbolic p2.

## Optional: self-training fine-tune here too

Independent of the served 27B: generate the verified set locally
(`selftrain_export`), upload it here, and QDoRA-train the **local 4B** on this
A100. Uploads `reasoning-selftrain.jsonl` + `qdora_deviant.py` + `requirements.txt`
from your machine (or clone the repo).

In [ ]:
# from google.colab import files; files.upload()   # reasoning-selftrain.jsonl + qdora_deviant.py + requirements.txt
# !pip -q install -r requirements.txt
# !python qdora_deviant.py reasoning-selftrain.jsonl --base-model Qwen/Qwen3-4B-Thinking-2507 --allow-small
# then download the adapter and convert to GGUF (see training/README.md)
print('uncomment the lines above to train; see training/README.md')

## Shutdown (run when done)

In [ ]:
for name in ['cf', 'srv']:
    try: globals()[name].terminate()
    except Exception: pass
print('stopped the tunnel and Ollama. Also: Runtime -> Disconnect and delete runtime.')